In [1]:
# Cell 1 — Install dependencies (run once in a fresh environment)
!pip -q install tensorflow tensorflow-datasets transformers accelerate evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports & hardware check
import platform
import tensorflow as tf
import tensorflow_datasets as tfds

from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices("GPU"))


Python version      : 3.12.12
TensorFlow version  : 2.19.0
GPU devices detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Cell 3 — Load the IMDB dataset (balanced 25k/25k, pre-split)
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)

print(ds_info)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.2A8R18_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.2A8R18_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.2A8R18_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.
tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='/root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
   

In [4]:
# Cell 4 — Peek at a couple of samples
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")


Label: Negative
This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline ...

Label: Negative
I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film wa ...



In [5]:
# Cell 5 — Tokenizer setup
MAX_LENGTH = 256   # Pad/trim reviews so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokenizer loaded: bert-base-uncased


In [6]:
# Cell 6 — Encoding helpers (HF tokenizer inside TF pipeline)
def encode_review(review_input):
    """
    Convert one review (bytes/tensor/str) to BERT inputs using Hugging Face tokenizer.
    Returns a dict with input_ids, attention_mask, token_type_ids as Python lists.
    """
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )


def tf_encode(text, label):
    """
    Wrap Python tokenization logic in tf.py_function so it can run in a tf.data pipeline.
    Produces a ({inputs}, label) pair compatible with TFBertForSequenceClassification.
    """
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32],
    )

    # Set static shapes so TF knows tensor ranks (important for Keras/graph mode)
    encoded[0].set_shape([MAX_LENGTH])  # input_ids
    encoded[1].set_shape([MAX_LENGTH])  # attention_mask
    encoded[2].set_shape([MAX_LENGTH])  # token_type_ids

    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2],
    }, label


def prepare_dataset(dataset, shuffle=True):
    """
    Build a performant input pipeline: tokenize -> (optional shuffle) -> batch -> prefetch.
    """
    ds = dataset.map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = prepare_dataset(ds_train, shuffle=True)
test_ds = prepare_dataset(ds_test, shuffle=False)


In [7]:
# Cell 7 — Initialize the fine-tuning model
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False,  # Avoid environments that lack safetensors support
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()


tf_model.h5:   0%|          | 0.00/536M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0 (unused)
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 109483778 (417.65 MB)
Trainable params: 109483778 (417.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
# Cell 8 — Train and monitor
EPOCHS = 2  # Increase to 3 if you have time/compute

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
)


Epoch 1/2
1563/1563 [==============================] - ETA: 0s - loss: 0.2716 - accuracy: 0.8869

KeyboardInterrupt: 

In [9]:
# Cell 9 — Evaluate on the held-out test set
eval_metrics = model.evaluate(test_ds, return_dict=True)
print("Test metrics:", eval_metrics)


1563/1563 [==============================] - 502s 321ms/step - loss: 0.2257 - accuracy: 0.9062
Test metrics: {'loss': 0.22573935985565186, 'accuracy': 0.9061599969863892}


In [10]:
# Cell 10 — Reusable inference helper
import numpy as np

def predict_sentiment(text: str):
    """
    Predict sentiment for a single input string.
    Returns:
      - label: "Positive" or "Negative"
      - confidence: float in [0, 1]
    """
    enc = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="tf",
    )

    outputs = model(enc, training=False)
    logits = outputs.logits  # shape: (1, 2)
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]  # shape: (2,)

    pred_id = int(np.argmax(probs))
    label = "Negative" if pred_id == 0 else "Positive"
    confidence = float(np.max(probs))
    return label, confidence


custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Prediction: Negative (confidence=0.500)


In [11]:
# Cell 11 — Quick sanity check on a few examples
samples = [
    "This movie was an absolute masterpiece. I loved every minute of it.",
    "Terrible plot and awful acting. I regret watching it.",
    "It was okay, not great but not terrible either.",
    "The customer support agent was rude and unhelpful. I'm very disappointed.",
]

for s in samples:
    lbl, conf = predict_sentiment(s)
    print(f"- {lbl} ({conf:.3f}) :: {s}")


- Positive (0.890) :: This movie was an absolute masterpiece. I loved every minute of it.
- Negative (0.992) :: Terrible plot and awful acting. I regret watching it.
- Negative (0.669) :: It was okay, not great but not terrible either.
- Negative (0.954) :: The customer support agent was rude and unhelpful. I'm very disappointed.
